# Lab 3b: VDB Corpus Acquisition with FAISS
**Course:** CIS 531/731 | **Module 3: Obtain**

## Objective
Take the text output generated by the Whisper ASR pipeline, convert it into semantic embeddings using a HuggingFace Transformer, and index it in a local Vector Database (FAISS).

### DevContainer Setup Instructions
Your `verify_env.py` script from MP2 already confirmed the presence of `faiss` and `transformers`. If your DevContainer failed to build those, rebuild the container now. If you are running locally outside the container, you must run:
`!pip install faiss-cpu transformers sentence-transformers`

In [7]:
import sys
import subprocess

required_packages = ["faiss-cpu", "transformers", "sentence-transformers"]
for package in required_packages:
    try:
        if package == "faiss-cpu":
            import faiss
        elif package == "transformers":
            import transformers
        elif package == "sentence-transformers":
            import sentence_transformers
    except ImportError:
        print(f"Installing missing package: {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"FAISS Version: {faiss.__version__}")

FAISS Version: 1.15.1


### 1. Initialize the Embedding Model
We use `all-MiniLM-L6-v2` as a lightweight, highly efficient sentence embedding model to encode our transcriptions into high-dimensional vectors.

In [3]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vector_dimension = embedder.get_sentence_embedding_dimension()
print(f"Model loaded. Embedding vector dimension: {vector_dimension}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding vector dimension: 384


/var/folders/1n/333g3g35337gs65lxlbhzkb80000gn/T/ipykernel_18062/3411531826.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  vector_dimension = embedder.get_sentence_embedding_dimension()


### 2. Create the FAISS Index & Ingest Corpus
Here we simulate the ingestion of chunks coming from our live radio stream pipeline.

In [4]:
# Initialize a flat L2 (Euclidean distance) FAISS index
index = faiss.IndexFlatL2(vector_dimension)

# Simulated incoming ASR corpus chunks
corpus = [
    "The caller sounds really upset about their recent breakup.",
    "I love this Cage the Elephant track, it reminds me of summer.",
    "Sometimes I feel like our whole generation is just out of time."
]

# Encode the text into vectors
corpus_embeddings = embedder.encode(corpus)

# Add vectors to the VDB
index.add(np.array(corpus_embeddings).astype('float32'))
print(f"Successfully ingested {index.ntotal} vectors into FAISS.")

Successfully ingested 3 vectors into FAISS.


### 3. Query the VDB (Mapping Subtle Tropes)
We query the database for a semantic concept ("doomed youth") to see which transcribed chunk matches closest in latent space.

In [5]:
query = "doomed youth"
query_embedding = embedder.encode([query]).astype('float32')

# Search the top 1 most similar result (k=1)
distances, indices = index.search(query_embedding, k=1)

matched_index = indices[0][0]
print(f"Query: '{query}'")
print(f"Closest Match in VDB: '{corpus[matched_index]}'")
print(f"L2 Distance: {distances[0][0]:.4f}")

Query: 'doomed youth'
Closest Match in VDB: 'Sometimes I feel like our whole generation is just out of time.'
L2 Distance: 1.2453


In [6]:
# Write the FAISS index to disk, reload it back into memory, and query again.
def persist_and_query_faiss(index_obj, query_text, index_filename="faiss_trope_index.bin"):
    # Save index to disk
    faiss.write_index(index_obj, index_filename)
    print(f"\n[CIS 731 Extension] Successfully wrote FAISS index to disk: {index_filename}")
    
    # Reload index from disk into memory
    loaded_index = faiss.read_index(index_filename)
    print(f"[CIS 731 Extension] Successfully reloaded FAISS index from disk. Total vectors: {loaded_index.ntotal}")
    
    # Encode query vector
    q_embedding = embedder.encode([query_text]).astype('float32')
    
    # Perform similarity search on the reloaded index
    dists, inds = loaded_index.search(q_embedding, k=1)
    return dists, inds, loaded_index

# Execute the disk persistence function
grad_distances, grad_indices, reloaded_index = persist_and_query_faiss(index, query)
grad_matched_index = grad_indices[0][0]

print(f"\n--- CIS 731 DISK I/O REQUERY RESULTS ---")
print(f"Query: '{query}'")
print(f"Closest Match from Reloaded VDB: '{corpus[grad_matched_index]}'")
print(f"L2 Distance: {grad_distances[0][0]:.4f}")


[CIS 731 Extension] Successfully wrote FAISS index to disk: faiss_trope_index.bin
[CIS 731 Extension] Successfully reloaded FAISS index from disk. Total vectors: 3

--- CIS 731 DISK I/O REQUERY RESULTS ---
Query: 'doomed youth'
Closest Match from Reloaded VDB: 'Sometimes I feel like our whole generation is just out of time.'
L2 Distance: 1.2453
